In [13]:
import json
import math
import os.path
from pathlib import Path
import torch
import numpy as np
import pandas as pd

from phi_3_5_constants import seed, dsets_folder, token_lengths_path, train_split_records_path, \
    validation_split_records_path

In [14]:
np_rng = np.random.default_rng(seed)
train_fraction = 0.8

In [15]:
source_path = Path("D:\\TruthIsUniversal_In_Phi_3_5_Mini\\all_layers_activations_for_final_token_of_sequences")

In [16]:
dsets_index_df = pd.read_csv(dsets_folder / "datasets_index.csv", index_col="Idx")

In [17]:
#dict of dset idx (in dsets_index_df) to list of record indices 
train_split_record_idxs: dict[int, list[int]]
validation_split_record_idxs: dict[int, list[int]]

#dict of dset idx (in dsets_index_df) to list of length-in-tokens for each record of that dataset
record_lengths_in_tokens: dict[int, list[int]]

In [18]:
if train_split_records_path.exists():
    with train_split_records_path.open("r") as f:
        train_split_record_idxs = json.load(f)
else:
    train_split_record_idxs = {}
if validation_split_records_path.exists():
    with validation_split_records_path.open("r") as f:
        validation_split_record_idxs = json.load(f)
else:
    validation_split_record_idxs = {}
if token_lengths_path.exists():
    with token_lengths_path.open("r") as f:
        record_lengths_in_tokens = json.load(f)
else:
    record_lengths_in_tokens = {}

In [19]:
for dset_idx, row in dsets_index_df.iterrows():
    categ_nm = row["Categ_Folder"]
    dset_file_nm = row["Dataset_File"]
    dset_nm = os.path.splitext(dset_file_nm)[0]
    harvest_result_path = source_path / categ_nm / (dset_nm + ".pt")
    
    if str(dset_idx) in record_lengths_in_tokens:
        continue
    
    activations_harvest_result = torch.load(harvest_result_path, weights_only=True)
    
    token_lengths = activations_harvest_result['token_lengths'].tolist()
    dset_size = len(token_lengths)    
    
    record_lengths_in_tokens[dset_idx] = token_lengths
    
    
    train_size = math.floor(dset_size*train_fraction)
    train_split_indices = sorted(np_rng.choice(dset_size, train_size, replace=False).tolist())
    validation_split_indices = sorted(list(set(range(dset_size)) - set(train_split_indices)))
    train_split_record_idxs[dset_idx] = train_split_indices
    validation_split_record_idxs[dset_idx] = validation_split_indices

In [20]:
with token_lengths_path.open("w") as f:
    json.dump(record_lengths_in_tokens, f)
with train_split_records_path.open("w") as f:
    json.dump(train_split_record_idxs, f)
with validation_split_records_path.open("w") as f:
    json.dump(validation_split_record_idxs, f)